# LeetCode 3620 — Network Recovery Pathways bằng Python

- Link: https://leetcode.com/problems/network-recovery-pathways/
- Độ khó: **Hard**
- Chủ đề: **Tìm nhị phân trên đáp án** (max–min), **đường đi ngắn nhất trên DAG** theo thứ tự topo, Dijkstra, lọc đồ thị theo ngưỡng

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findMaxPathScore(self, edges: List[List[int]], online: List[bool], k: int) -> int:
```

## 1. Mục tiêu học

1. "Tối đa hoá cạnh nhỏ nhất" ⇒ câu hỏi quyết định: **có đường đi hợp lệ mà mọi cạnh có chi phí `≥ t` không?**
2. Với ngưỡng `t` cố định, đường hợp lệ tồn tại ⇔ **đường ngắn nhất** (tổng chi phí) từ `0` tới `n−1` trong đồ thị con (cạnh `≥ t`, đỉnh online) `≤ k`.
3. Tính đơn điệu theo `t` ⇒ tìm nhị phân trên danh sách chi phí đã sắp.
4. Đường ngắn nhất trên **DAG**: duyệt thứ tự topo một lần `O(n + m)` — không cần Dijkstra.
5. Kiểm chứng bằng liệt kê mọi đường đi trên DAG nhỏ.

## 2. Đề bài & ví dụ

DAG `n` đỉnh, cạnh `[u, v, cost]`. Một số đỉnh offline (đỉnh `0`, `n−1` luôn online). Đường `0 → n−1` hợp lệ nếu mọi đỉnh trung gian online và tổng chi phí `≤ k`. Điểm của đường = cạnh **nhỏ nhất** trên đường. Trả điểm lớn nhất, hoặc `-1`.

**Ví dụ 1:** `edges = [[0,1,5],[1,3,10],[0,2,3],[2,3,4]]`, tất cả online, `k = 10` → `3`.

**Ví dụ 2:** `edges = [[0,1,7],[1,4,5],[0,2,6],[2,3,6],[3,4,2],[2,4,6]]`, `online = [T,T,T,F,T]`, `k = 12` → `6`.

**Ràng buộc:** `n ≤ 5·10^4`, `m ≤ 10^5`, `cost ≤ 10^9`, `k ≤ 5·10^{13}`.

In [1]:
from typing import List
from collections import deque
import heapq, random, time

examples = [
    ([[0, 1, 5], [1, 3, 10], [0, 2, 3], [2, 3, 4]], [True] * 4, 10, 3),
    ([[0, 1, 7], [1, 4, 5], [0, 2, 6], [2, 3, 6], [3, 4, 2], [2, 4, 6]], [True, True, True, False, True], 12, 6),
]
for e, on, k, exp in examples:
    print(e, on, k, "-> kỳ vọng", exp)

[[0, 1, 5], [1, 3, 10], [0, 2, 3], [2, 3, 4]] [True, True, True, True] 10 -> kỳ vọng 3
[[0, 1, 7], [1, 4, 5], [0, 2, 6], [2, 3, 6], [3, 4, 2], [2, 4, 6]] [True, True, True, False, True] 12 -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1. Thứ tự topo (Kahn)

Lặp lấy đỉnh có bậc vào 0. Trên DAG luôn ra đủ `n` đỉnh, và mọi cạnh `u → v` có `u` đứng trước `v`.

In [2]:
def topo_order(n, edges):
    adj = [[] for _ in range(n)]; indeg = [0] * n
    for u, v, c in edges: adj[u].append((v, c)); indeg[v] += 1
    q = deque(i for i in range(n) if indeg[i] == 0); order = []
    while q:
        u = q.popleft(); order.append(u)
        for v, _ in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0: q.append(v)
    return adj, order

print(topo_order(4, examples[0][0])[1])

[0, 1, 2, 3]


### 3.2. Đường ngắn nhất trên DAG

`dist[0] = 0`; duyệt đỉnh theo thứ tự topo, thư giãn mọi cạnh ra. Đúng vì khi xử lý `u`, mọi đường tới `u` đã được xét.

### 3.3. Đơn điệu của câu hỏi quyết định

`ok(t)` = "có đường hợp lệ với mọi cạnh `≥ t`". Nếu `ok(t)` đúng thì `ok(t')` đúng với mọi `t' ≤ t` (đồ thị con lớn hơn). Đáp án = `t` lớn nhất (trong các giá trị chi phí) thoả `ok`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Điểm của đường `≥ t` ⇔ mọi cạnh của đường `≥ t`. Trong số các đường như vậy, đường **rẻ nhất** có cơ hội thoả `≤ k` cao nhất ⇒ chỉ cần kiểm tra đường ngắn nhất.

Ví dụ 2 với `t = 6`: các cạnh `≥ 6` giữa đỉnh online: `0→1 (7)`, `0→2 (6)`, `2→4 (6)` (cạnh `2→3` bị loại vì 3 offline). Đường ngắn nhất `0 → 2 → 4` tổng 12 `≤ 12` ⇒ ok. `t = 7`: chỉ còn `0→1`, không tới được 4 ⇒ không ok ⇒ đáp án `6`. ✔

## 5. Cách 1 — Brute Force: liệt kê mọi đường

In [3]:
def brute(edges, online, k):
    n = len(online); adj = [[] for _ in range(n)]
    for u, v, c in edges: adj[u].append((v, c))
    best = -1
    def dfs(u, total, mn):
        nonlocal best
        if total > k: return
        if u == n - 1: best = max(best, mn); return
        for v, c in adj[u]:
            if v != n - 1 and not online[v]: continue
            dfs(v, total + c, min(mn, c))
    dfs(0, 0, float("inf"))
    return best if best != float("inf") else -1

for e, on, k, exp in examples:
    print(brute(e, on, k), "kỳ vọng", exp)

3 kỳ vọng 3
6 kỳ vọng 6


**Nút thắt:** số đường trong DAG có thể theo hàm mũ.

## 6. Ý tưởng thuật toán / bước đột phá

1. Tính thứ tự topo **một lần** (trên toàn đồ thị — thứ tự vẫn hợp lệ cho mọi đồ thị con).
2. `ok(t)`: DP đường ngắn nhất theo thứ tự topo, chỉ dùng cạnh `c ≥ t` và đỉnh đích online (hoặc là `n−1`); trả `dist[n−1] ≤ k`.
3. Tìm nhị phân trên danh sách chi phí phân biệt đã sắp: `O((n + m) log m)`.

## 7. Các cách tiếp theo

- **Cách 2:** thử **mọi** ngưỡng (tuyến tính) — `O(m (n + m))`.
- **Cách 3:** tìm nhị phân + DP trên DAG (lời giải nộp).
- **Cách 4:** tìm nhị phân + **Dijkstra** (không dùng tính chất DAG) — `O((n + m) log n · log m)`.

### 7.1. Cách 2 — Thử mọi ngưỡng

In [4]:
INF = float("inf")

def make_ok(edges, online, k):
    n = len(online); adj, order = topo_order(n, edges)
    def ok(t):
        dist = [INF] * n; dist[0] = 0
        for u in order:
            du = dist[u]
            if du == INF: continue
            for v, c in adj[u]:
                if c >= t and (online[v] or v == n - 1) and du + c < dist[v]: dist[v] = du + c
        return dist[n - 1] <= k
    return ok

def linear_thresholds(edges, online, k):
    ok = make_ok(edges, online, k); best = -1
    for t in sorted({c for _, _, c in edges}):
        if ok(t): best = t
    return best

for e, on, k, exp in examples:
    print(linear_thresholds(e, on, k), "kỳ vọng", exp)

3 kỳ vọng 3
6 kỳ vọng 6


### 7.2. Cách 3 — Tìm nhị phân + DP trên DAG (lời giải chính)

In [5]:
def binary_dag(edges, online, k):
    ok = make_ok(edges, online, k)
    vals = sorted({c for _, _, c in edges})
    if not vals or not ok(vals[0]): return -1
    lo, hi = 0, len(vals) - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(vals[mid]): lo = mid
        else: hi = mid - 1
    return vals[lo]

for e, on, k, exp in examples:
    print(binary_dag(e, on, k), "kỳ vọng", exp)

3 kỳ vọng 3
6 kỳ vọng 6


### 7.3. Cách 4 — Tìm nhị phân + Dijkstra

In [6]:
def binary_dijkstra(edges, online, k):
    n = len(online); adj = [[] for _ in range(n)]
    for u, v, c in edges: adj[u].append((v, c))
    def ok(t):
        dist = [INF] * n; dist[0] = 0; pq = [(0, 0)]
        while pq:
            d, u = heapq.heappop(pq)
            if d > dist[u]: continue
            if u == n - 1: return d <= k
            for v, c in adj[u]:
                if c >= t and (online[v] or v == n - 1) and d + c < dist[v]: dist[v] = d + c; heapq.heappush(pq, (d + c, v))
        return False
    vals = sorted({c for _, _, c in edges})
    if not vals or not ok(vals[0]): return -1
    lo, hi = 0, len(vals) - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(vals[mid]): lo = mid
        else: hi = mid - 1
    return vals[lo]

for e, on, k, exp in examples:
    print(binary_dijkstra(e, on, k), "kỳ vọng", exp)

3 kỳ vọng 3
6 kỳ vọng 6


## 8. TRACE từng bước

In [7]:
def trace(edges, online, k):
    n = len(online); adj, order = topo_order(n, edges); vals = sorted({c for _, _, c in edges})
    print("thứ tự topo:", order, "| các ngưỡng:", vals)
    for t in vals:
        dist = [INF] * n; dist[0] = 0
        for u in order:
            if dist[u] == INF: continue
            for v, c in adj[u]:
                if c >= t and (online[v] or v == n - 1): dist[v] = min(dist[v], dist[u] + c)
        d = dist[n - 1]
        print(f"  t = {t}: đường ngắn nhất tới {n - 1} = {d} -> {'ok' if d <= k else 'không'}")

trace(*examples[1][:3])

thứ tự topo: [0, 1, 2, 3, 4] | các ngưỡng: [2, 5, 6, 7]
  t = 2: đường ngắn nhất tới 4 = 12 -> ok
  t = 5: đường ngắn nhất tới 4 = 12 -> ok
  t = 6: đường ngắn nhất tới 4 = 12 -> ok
  t = 7: đường ngắn nhất tới 4 = inf -> không


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def findMaxPathScore(self, edges: List[List[int]], online: List[bool], k: int) -> int:
        n = len(online)
        adj = [[] for _ in range(n)]
        indeg = [0] * n
        for u, v, c in edges:
            if (online[u] or u == 0) and (online[v] or v == n - 1):
                adj[u].append((v, c))
                indeg[v] += 1
        order = []
        q = deque(i for i in range(n) if indeg[i] == 0)
        while q:
            u = q.popleft()
            order.append(u)
            for v, _ in adj[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        INF = float('inf')

        def ok(t):
            dist = [INF] * n
            dist[0] = 0
            for u in order:
                du = dist[u]
                if du == INF:
                    continue
                for v, c in adj[u]:
                    if c >= t and du + c < dist[v]:
                        dist[v] = du + c
            return dist[n - 1] <= k

        vals = sorted({c for _, _, c in edges})
        if not vals or not ok(vals[0]):
            return -1
        lo, hi = 0, len(vals) - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if ok(vals[mid]):
                lo = mid
            else:
                hi = mid - 1
        return vals[lo]

sol = Solution()
for e, on, k, exp in examples:
    got = sol.findMaxPathScore(e, on, k)
    print(got, "✔" if got == exp else "✘")

3 ✔
6 ✔


## 10. Vì sao thuật toán đúng

1. **Quyết định:** tồn tại đường hợp lệ có điểm `≥ t` ⇔ tồn tại đường `0 → n−1` qua đỉnh online, mọi cạnh `≥ t`, tổng `≤ k` ⇔ đường ngắn nhất trong đồ thị con `G_t` có tổng `≤ k`.
2. **DP trên DAG** cho đúng đường ngắn nhất (thứ tự topo của đồ thị gốc vẫn là thứ tự topo của đồ thị con).
3. **Đơn điệu:** `G_{t'} ⊇ G_t` khi `t' ≤ t` ⇒ `ok` không tăng theo `t`.
4. **Đáp án thuộc tập chi phí:** điểm của mọi đường là một chi phí cạnh ⇒ tìm `t` lớn nhất trong tập chi phí phân biệt có `ok(t)`. Nếu `ok(min)` sai ⇒ không có đường hợp lệ nào ⇒ `-1`. (Trường hợp `n − 1` không có cạnh vào ⇒ `-1`.)
5. Đỉnh offline bị loại bằng cách bỏ mọi cạnh vào nó (và ra khỏi nó).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê đường | hàm mũ | `O(n)` |
| 2. Thử mọi ngưỡng | `O(m (n + m))` | `O(n + m)` |
| 3. Nhị phân + DAG | `O((n + m) log m)` | `O(n + m)` |
| 4. Nhị phân + Dijkstra | `O((n + m) log n log m)` | `O(n + m)` |

## 12. Bộ test

In [9]:
METHODS = [brute, linear_thresholds, binary_dag, binary_dijkstra, lambda e, o, k: Solution().findMaxPathScore(e, o, k)]
for e, on, k, exp in examples:
    assert all(f(e, on, k) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [10]:
# 12.2 ca biên
assert Solution().findMaxPathScore([], [True, True], 5) == -1                         # không có cạnh
assert Solution().findMaxPathScore([[0, 1, 0]], [True, True], 0) == 0                 # chi phí 0
assert Solution().findMaxPathScore([[0, 1, 9]], [True, True], 8) == -1                # vượt k
assert Solution().findMaxPathScore([[0, 1, 5], [1, 2, 5]], [True, False, True], 100) == -1   # đi qua đỉnh offline
print("Ca biên: OK")

Ca biên: OK


In [11]:
# 12.3 DAG ngẫu nhiên
def rand_dag(rng, n, p):
    perm = list(range(1, n - 1)); rng.shuffle(perm); order = [0] + perm + [n - 1]
    edges = []
    for i in range(n):
        for j in range(i + 1, n):
            if rng.random() < p: edges.append([order[i], order[j], rng.randint(0, 9)])
    return edges

rng = random.Random(3620)
for _ in range(800):
    n = rng.randint(2, 7); e = rand_dag(rng, n, rng.random())
    on = [True] + [rng.random() < 0.8 for _ in range(n - 2)] + [True] if n > 2 else [True, True]
    k = rng.randint(0, 25)
    b = brute(e, on, k)
    assert all(f(e, on, k) == b for f in METHODS[1:]), (e, on, k)
print("800 test ngẫu nhiên: OK")

800 test ngẫu nhiên: OK


In [12]:
# 12.4 test lớn: n = 5·10^4, m = 10^5
rng = random.Random(1)
n = 5 * 10**4; perm = list(range(1, n - 1)); rng.shuffle(perm); order = [0] + perm + [n - 1]
edges = []
for i in range(n - 1): edges.append([order[i], order[i + 1], rng.randint(0, 10**9)])
while len(edges) < 10**5:
    i = rng.randrange(n - 1); j = rng.randrange(i + 1, min(n, i + 50)); edges.append([order[i], order[j], rng.randint(0, 10**9)])
online = [True] + [rng.random() < 0.95 for _ in range(n - 2)] + [True]
k = 5 * 10**13
t0 = time.time(); x = Solution().findMaxPathScore(edges, online, k); t1 = time.time()
print("đáp án:", x, "| thời gian:", round(t1 - t0, 2), "s")

đáp án: 208534981 | thời gian: 0.44 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tìm đường có cạnh nhỏ nhất lớn nhất (bỏ qua giới hạn `k`)

In [13]:
def wrong_ignore_k(edges, online, k):
    return brute(edges, online, float("inf"))                                       # SAI

print("sai:", wrong_ignore_k(*examples[0][:3]), "| đúng:", brute(*examples[0][:3]))

sai: 5 | đúng: 3


### Lỗi 2 — Với ngưỡng `t` chỉ kiểm tra **có đường** (BFS), không kiểm tra tổng `≤ k`

In [14]:
def wrong_reachability(edges, online, k):
    n = len(online); best = -1
    for t in sorted({c for _, _, c in edges}):
        adj = [[] for _ in range(n)]
        for u, v, c in edges:
            if c >= t and (online[v] or v == n - 1): adj[u].append(v)
        seen = {0}; st = [0]
        while st:
            u = st.pop()
            for v in adj[u]:
                if v not in seen: seen.add(v); st.append(v)
        if n - 1 in seen: best = t                                                    # SAI: thiếu điều kiện k
    return best

print("sai:", wrong_reachability(*examples[0][:3]), "| đúng:", brute(*examples[0][:3]))

sai: 5 | đúng: 3


### Lỗi 3 — Không loại đỉnh offline

In [15]:
def wrong_offline(edges, online, k):
    return binary_dag(edges, [True] * len(online), k)                               # SAI

e, on, k, exp = examples[1]
e2 = e + [[0, 3, 9], [3, 4, 9]]
print("sai:", wrong_offline(e2, on, k + 6), "| đúng:", brute(e2, on, k + 6))

sai: 9 | đúng: 6


### Lỗi 4 — Tìm nhị phân trên `[0, 10^9]` với điều kiện sai biên (vòng lặp vô hạn / lệch 1)

Khi dùng `mid = (lo + hi) // 2` cùng `lo = mid`, vòng lặp có thể kẹt khi `hi = lo + 1`. Với "tìm giá trị lớn nhất thoả", dùng `mid = (lo + hi + 1) // 2`.

In [16]:
lo, hi, steps = 0, 1, 0
while lo < hi and steps < 5:
    mid = (lo + hi) // 2; lo = mid; steps += 1                                      # SAI: kẹt ở lo = 0
print("mid làm tròn xuống kẹt sau", steps, "bước với lo =", lo, "hi =", hi)

mid làm tròn xuống kẹt sau 5 bước với lo = 0 hi = 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối đa hoá giá trị nhỏ nhất trên đường" + ràng buộc phụ (tổng chi phí) ⇒ tìm nhị phân ngưỡng + kiểm tra bằng đường ngắn nhất.
- DAG ⇒ đường ngắn nhất bằng thứ tự topo, `O(n + m)`, kể cả cạnh chi phí 0.
- Đỉnh bị cấm ⇒ lọc cạnh một lần trước khi tìm nhị phân.
- Đáp án luôn nằm trong tập chi phí ⇒ tìm nhị phân trên danh sách đã sắp (ít bước hơn).

## 15. Biến thể của bài

1. **Không có giới hạn `k`**: bài "bottleneck path" — có thể dùng Kruskal ngược hoặc DP max–min trên DAG trực tiếp.
2. **Đồ thị có chu trình**: thay DP topo bằng Dijkstra (Cách 4).
3. **Tối đa hoá cạnh nhỏ nhất với tối đa `L` cạnh**: kiểm tra bằng BFS theo số cạnh.
4. **Tối thiểu hoá cạnh lớn nhất** (minimax): tìm nhị phân ngưỡng trên, chỉ dùng cạnh `≤ t`.

In [17]:
# Biến thể 1: bottleneck path trên DAG (không giới hạn k) bằng DP max-min
def bottleneck_dag(edges, online, n):
    adj, order = topo_order(n, edges); best = [-1] * n; best[0] = float("inf")
    for u in order:
        if best[u] == -1: continue
        for v, c in adj[u]:
            if online[v] or v == n - 1: best[v] = max(best[v], min(best[u], c))
    return best[n - 1] if best[n - 1] != float("inf") else -1

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(2, 7); e = rand_dag(rng, n, rng.random())
    on = [True] + [rng.random() < 0.8 for _ in range(n - 2)] + [True] if n > 2 else [True, True]
    assert bottleneck_dag(e, on, n) == brute(e, on, float("inf"))
print("Biến thể 1 OK:", bottleneck_dag(examples[0][0], examples[0][1], 4))

Biến thể 1 OK: 5


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Liệt kê đường | DFS | hàm mũ | Kiểm chứng |
| 2. Mọi ngưỡng | Kiểm tra tuyến tính | `O(m(n+m))` | `m` nhỏ |
| 3. Nhị phân + DAG | Đơn điệu + topo | `O((n+m) log m)` | **Lời giải chuẩn** |
| 4. Nhị phân + Dijkstra | Tổng quát | `O((n+m) log² )` | Đồ thị có chu trình |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `edges = [[0,1,4],[1,2,4],[0,2,1]]`, mọi đỉnh online, `k = 7`.

**Bài 2 (tính tay).** Cùng đồ thị, `k = 8`.

**Bài 3 (điền TODO).** Viết `min_cost_with_score(edges, online, t)`: tổng chi phí nhỏ nhất của đường có điểm `≥ t` (−1 nếu không có).

**Bài 4 (tự viết lại).** Viết lại `ok(t)` bằng **đệ quy có nhớ** trên DAG (đường ngắn nhất từ `u` tới `n−1`).

**Bài 5 (mini challenge).** Trả về luôn **một đường** đạt đáp án.

In [18]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def min_cost_with_score(edges, online, t):
    # TODO
    pass

In [19]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def best_path(edges, online, k):
    # TODO: tìm t* rồi truy vết đường ngắn nhất trong G_{t*}
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đường `0→1→2`: tổng 8 > 7. (2) Đường `0→2`: tổng 1, điểm 1. (3) `1`.

**Bài 2.** (1) `k = 8` cho phép `0→1→2` (điểm 4). (2) Lớn hơn 1. (3) `4`.

**Bài 3.** (1) Lọc cạnh `≥ t`. (2) DP topo. (3) Trả `dist[n−1]` hoặc −1.

**Bài 4.** (1) `f(u) = min_{u→v, c ≥ t} c + f(v)`. (2) `f(n−1) = 0`. (3) `lru_cache`, chú ý độ sâu đệ quy với DAG dài.

**Bài 5.** (1) `t*` từ lời giải. (2) DP topo trong `G_{t*}` lưu cha. (3) Truy vết từ `n−1`.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2
g = [[0, 1, 4], [1, 2, 4], [0, 2, 1]]
assert Solution().findMaxPathScore(g, [True] * 3, 7) == 1
assert Solution().findMaxPathScore(g, [True] * 3, 8) == 4
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [21]:
# Bài 3, 4
from functools import lru_cache
def min_cost_with_score_sol(edges, online, t):
    n = len(online); adj, order = topo_order(n, edges); dist = [INF] * n; dist[0] = 0
    for u in order:
        if dist[u] == INF: continue
        for v, c in adj[u]:
            if c >= t and (online[v] or v == n - 1): dist[v] = min(dist[v], dist[u] + c)
    return dist[n - 1] if dist[n - 1] < INF else -1

def ok_memo(edges, online, k, t):
    n = len(online); adj = [[] for _ in range(n)]
    for u, v, c in edges: adj[u].append((v, c))
    @lru_cache(None)
    def f(u):
        if u == n - 1: return 0
        return min((c + f(v) for v, c in adj[u] if c >= t and (online[v] or v == n - 1)), default=INF)
    return f(0) <= k

rng = random.Random(4)
for _ in range(300):
    n = rng.randint(2, 7); e = rand_dag(rng, n, rng.random())
    on = [True] + [rng.random() < 0.8 for _ in range(n - 2)] + [True] if n > 2 else [True, True]
    t = rng.randint(0, 9); k = rng.randint(0, 25)
    assert ok_memo(e, on, k, t) == make_ok(e, on, k)(t)
    mc = min_cost_with_score_sol(e, on, t)
    assert (mc != -1 and mc <= k) == make_ok(e, on, k)(t)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [22]:
# Bài 5
def best_path_sol(edges, online, k):
    t = Solution().findMaxPathScore(edges, online, k)
    if t == -1: return None
    n = len(online); adj, order = topo_order(n, edges); dist = [INF] * n; par = [-1] * n; dist[0] = 0
    for u in order:
        if dist[u] == INF: continue
        for v, c in adj[u]:
            if c >= t and (online[v] or v == n - 1) and dist[u] + c < dist[v]: dist[v] = dist[u] + c; par[v] = u
    path = [n - 1]
    while path[-1] != 0: path.append(par[path[-1]])
    return t, path[::-1], dist[n - 1]

print("Bài 5:", best_path_sol(*examples[1][:3]))
res = best_path_sol(*examples[1][:3]); assert res[0] == 6 and res[2] <= 12

Bài 5: (6, [0, 2, 4], 12)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
lọc cạnh vào/ra đỉnh offline; thứ tự topo (Kahn)
ok(t): dist theo topo với cạnh c >= t; return dist[n-1] <= k
vals = sorted(set(cost)); if not ok(vals[0]): return -1
tìm t lớn nhất trong vals với ok(t)   (mid = (lo + hi + 1) // 2)
```

**Checklist:**

- [ ] Kiểm tra **tổng ≤ k** chứ không chỉ "tới được"?
- [ ] Loại đỉnh offline (trừ `0`, `n−1`)?
- [ ] Tìm nhị phân "lớn nhất thoả" làm tròn **lên**?
- [ ] Không có cạnh / không tới được ⇒ `-1`?
- [ ] Đã so với liệt kê đường trên DAG nhỏ?